In [ ]:
# 03_violation_baseline.ipynb -- LightGBM baseline for the 1-4-slot-lead-time
# frequency-violation target
# !pip install lightgbm -q

import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve

import features as f

TARGET = "violation_lead"

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
feat_df = f.build_feature_table(scada)

df = feat_df.dropna(subset=[TARGET]).copy()  # drop rows whose lead window is unresolvable

# --- Time-aware split ---
train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
test = df[df["date"] >= "2026-01-01"]

print("train:", train.shape, "val:", val.shape, "test:", test.shape)
print("event rate train/val/test:", train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean())

X_train, y_train = train[f.FEATURE_COLS], train[TARGET]
X_val, y_val = val[f.FEATURE_COLS], val[TARGET]
X_test, y_test = test[f.FEATURE_COLS], test[TARGET]

# NOTE (2026-07-11): deliberately NOT using scale_pos_weight here -- see
# features.py's scale_pos_weight() docstring for the full story. In short: it
# caused LightGBM's early stopping to fire after a single boosting round
# (best_iteration_=1) for this ~2-3% positive-rate target, in every configuration
# tried, silently training something close to a single shallow tree instead of
# the intended up-to-500-round ensemble. Removing it and using average_precision
# (not the default binary_logloss) as the early-stopping metric fixed this --
# best_iteration_ now lands around 50, and PR-AUC rose from 0.0614 to 0.0937.
model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="average_precision",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)
print(f"\nbest_iteration_: {model.best_iteration_}  (sanity check -- should NOT be 1)")

proba_test = model.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, proba_test)
base_rate = y_test.mean()
print(f"PR-AUC: {pr_auc:.4f}  (random baseline = base rate = {base_rate:.4f})")

preds_05 = (proba_test >= 0.5).astype(int)
print(f"F1 @ 0.5 threshold: {f1_score(y_test, preds_05):.4f}")

# A fixed 0.5 threshold is the wrong lens for a ~2-3% positive rate -- it almost never
# fires. Report the best-F1 operating point too, which is what an actual early-warning
# system would be tuned to.
precision, recall, thresh = precision_recall_curve(y_test, proba_test)
f1s = 2 * precision * recall / (precision + recall + 1e-12)
best_idx = np.nanargmax(f1s[:-1])
print(f"Best-F1 operating point: F1={f1s[best_idx]:.4f} at threshold={thresh[best_idx]:.4f} "
      f"(precision={precision[best_idx]:.4f}, recall={recall[best_idx]:.4f})")

idx95 = np.where(precision[:-1] >= 0.95)[0]
recall_at_95p = recall[idx95].max() if len(idx95) else 0.0
print(f"Recall at >=95% precision: {recall_at_95p:.4f}")

importance = pd.Series(model.feature_importances_, index=f.FEATURE_COLS).sort_values(ascending=False)
print("\nTop 15 features:\n", importance.head(15))

# --- Results (verified 2026-07-11, LightGBM 4.6.0; corrected after finding the
#     scale_pos_weight bug above -- these numbers supersede the first version of
#     this notebook, which reported PR-AUC 0.0614 / best-F1 0.1297) ---
# best_iteration_: 51 (not 1 -- confirms the model is actually training, not stalling)
# PR-AUC 0.0937 vs a random/base-rate baseline of 0.0305 -- 3.07x lift over chance
# (was 2.01x before the fix). F1@0.5 = 0.0207 (still near-zero -- 0.5 is still the
# wrong threshold for a ~3% positive rate). Best-F1 operating point (threshold ~0.09):
# F1=0.1712, precision=12.1%, recall=29.0% -- both PR-AUC and best-F1 are meaningfully
# better than the pre-fix numbers, though the precision/recall balance shifted (higher
# precision, lower recall than before -- catches fewer total violations but each flagged
# one is more often real). Recall at >=95% precision is still 0 -- a high-confidence-only
# alert mode remains out of reach with this feature set.
#
# Two real changes landed together here, both empirically verified before adoption:
# (1) the scale_pos_weight removal above (the larger effect), and (2) two new
# solar-generation-volatility features (solar_delta_mw, solar_roll8_std) added to
# features.py after an explicit diagnostic found the ORIGINAL hypothesis for this
# notebook -- that a demand-side ramp-shock (ramp_lead) precedes a violation -- was
# wrong: P(ramp_lead=1 | violation_lead=1) = 14.2%, actually BELOW the 15.1%
# unconditional rate. Scanning every generation source's slot-to-slot delta for
# correlation with violation_lead instead found solar clearly ahead of the rest (0.0785
# vs wind's 0.036, demand's ~0), consistent with violations clustering at 08:00-09:00
# and 13:00 (01_eda.ipynb) -- prime solar-ramp hours. Physically: a violation may be a
# downstream consequence of fast-changing SOLAR generation outrunning reserves, not
# fast-changing demand -- a different, better-supported hypothesis than the one this
# notebook originally shipped with.


In [ ]:
# --- Appendix: shorter lead-window experiment (2026-07-11, re-verified with the
#     corrected training config above -- no scale_pos_weight, average_precision
#     early-stopping metric, patience 50) ---
# Does shrinking the lookahead window from 1-4 slots improve the violation classifier?
# features.py's add_violation_label() and build_feature_table() both take a lead_slots
# override for exactly this test.

import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import average_precision_score, precision_recall_curve

import features as f

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
resid = f.build_study1_residual_signal()  # computed once, reused across all window sizes


def run(lead_slots):
    feat = f.build_feature_table(scada, study1_residual=resid, violation_lead_slots=lead_slots)
    df = feat.dropna(subset=["violation_lead"]).copy()
    train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
    val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
    test = df[df["date"] >= "2026-01-01"]

    X_train, y_train = train[f.FEATURE_COLS], train["violation_lead"]
    X_val, y_val = val[f.FEATURE_COLS], val["violation_lead"]
    X_test, y_test = test[f.FEATURE_COLS], test["violation_lead"]

    model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="average_precision",
              callbacks=[lgb.early_stopping(50, verbose=False)])

    proba = model.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, proba)
    base_rate = y_test.mean()

    precision, recall, thresh = precision_recall_curve(y_test, proba)
    f1s = 2 * precision * recall / (precision + recall + 1e-12)
    best_idx = np.nanargmax(f1s[:-1])

    print(f"lead_slots={lead_slots}: best_iter={model.best_iteration_}  base_rate={base_rate:.4f}  "
          f"PR-AUC={pr_auc:.4f} (lift={pr_auc / base_rate:.2f}x)  best-F1={f1s[best_idx]:.4f} "
          f"(P={precision[best_idx]:.4f} R={recall[best_idx]:.4f})")


for k in [4, 3, 2, 1]:
    run(k)

# --- Findings (re-verified 2026-07-11 with the corrected training config -- these
#     numbers supersede the first version of this appendix, which used the buggy
#     scale_pos_weight setup and understated every window's performance) ---
# lead_slots=4 (shipped, 15-60 min): PR-AUC=0.0937 (3.07x lift)  best-F1=0.1712 (P=12.1% R=29.0%)
# lead_slots=3 (15-45 min):          PR-AUC=0.0946 (3.84x lift)  best-F1=0.1649 (P=12.0% R=26.6%)
# lead_slots=2 (15-30 min):          PR-AUC=0.1254 (6.92x lift)  best-F1=0.1849 (P=24.1% R=15.0%)
# lead_slots=1 (15 min only):        PR-AUC=0.0985 (8.88x lift)  best-F1=0.1687 (P=12.9% R=24.5%)
#
# Different conclusion than the first (buggy) pass: this is no longer a flat trade-off
# with no winner. lead_slots=2 now has BOTH the highest PR-AUC (0.1254, well above 4's
# 0.0937) and the highest best-F1 (0.1849) of all four windows -- a much stronger case
# than before. The real trade-off that remains is precision vs. recall at that specific
# window: 2 slots trades raw recall (15.0%, catching fewer total violations) for much
# higher precision (24.1%, each flagged one more often real) than the shipped 4-slot
# default (12.1% precision, 29.0% recall). Whether to actually SHIP a shorter window is
# still a product decision (fewer, higher-confidence 30-min warnings vs. more, noisier
# 60-min warnings) rather than a pure modelling one, so the shipped default in
# predict.py remains 4 slots (matching the roadmap's original 1-4-slot scope) -- but
# the case for revisiting that default is now considerably stronger than the original
# (bug-affected) version of this experiment suggested.
